In [1]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import shap
from xgboost import XGBClassifier
# import random forest regressor
from sklearn.ensemble import RandomForestRegressor
#import linear regression
from sklearn.linear_model import LinearRegression
# import tqdm
from tqdm import tqdm
import tqdm
#import r2_score
from sklearn.metrics import r2_score
#import confusion matrix
from sklearn.metrics import confusion_matrix
# import roc auc score
from sklearn.metrics import roc_auc_score
from ipcch.food_crisis_functions import *
import json
with open("forecasting_hyperparameters.json", "r") as file:
    best_params_xgb_regressor= json.load(file)
    
with open("forecasting_hyperparameters_p3.json", "r") as file:
    best_params_xgb_regressor_for_p3= json.load(file)


In [2]:

# read csv
df = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\forecasting_subset_IPCCH_v1210.csv')
Nigeria_area = pd.read_csv(r'C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\Nigeria_codebook.csv')
# add dummys for area_id and month year
#df = pd.concat([df, pd.get_dummies(df['area_id'], prefix='area_id')], axis=1)
#df = pd.concat([df, pd.get_dummies(df['date'], prefix='month_year')], axis=1)
# drop lat and lon
#df = df.drop(['lat', 'lon'], axis=1)
###drop fews_ipc_ha
#df = df.drop(['fews_ipc_ha'], axis=1)
# random split train and test

# prepare date from year and month
df['date'] = pd.to_datetime(df[['year', 'month']].assign(DAY=1))
# check for inf and replace with na
df = df.replace([np.inf, -np.inf], np.nan)

# replace df['overall_phase], 0 as 1, >5 as 5
df['overall_phase'] = df['overall_phase'].apply(lambda x: 1 if x < 1 else (5 if x > 5 else x))

df_origin = df.copy()

# first group by area_id then by date
df = df.groupby(['area_id', 'date']).sum().reset_index()
#within each area_id, calculate the difference of days from last observation
df['diff_days'] = df.groupby('area_id')['date'].diff().dt.days
# group by area_id and date, create lagged 1 term of overall_phase, phase1_percent, phase2_percent, phase3_percent, phase4_percent, phase5_percent
df[['overall_phase_lag1', 'phase1_percent_lag1', 'phase2_percent_lag1', 'phase3_percent_lag1', 'phase4_percent_lag1', 'phase5_percent_lag1']] = df.groupby('area_id')[['overall_phase', 'phase1_percent', 'phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent']].shift(1)
#lag 2
df[['overall_phase_lag2', 'phase1_percent_lag2', 'phase2_percent_lag2', 'phase3_percent_lag2', 'phase4_percent_lag2', 'phase5_percent_lag2']] = df.groupby('area_id')[['overall_phase', 'phase1_percent', 'phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent']].shift(2)
#if diff_days<365, for lag1 use lag2, for lag2 shift one more term
outcome_var = ['overall_phase', 'phase1_percent', 'phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent']
for vars in outcome_var:
    df.loc[df['diff_days'] < 365, f'{vars}_lag1'] = df[f'{vars}_lag2']
    df.loc[df['diff_days'] < 365, f'{vars}_lag2'] = df.groupby('area_id')[vars].shift(3)

# drop diff_days
df.drop(columns=['diff_days'], inplace=True)
df_origin = df.copy()

In [3]:

y_pred_test = pd.DataFrame()
model_stats = pd.DataFrame()
#select phase1_percent is not na
df = df_origin[df_origin['phase1_percent'].notna()]

# Sort by region and date
df = df.sort_values(by=['area_id', 'date'])
#drop overall phase
df = df.drop(['overall_phase'], axis=1)
#for each region, set last observation to be test set
# create a series of new outcome, phase2_worse=phase2_percent+phase3_percent+phase4_percent+phase5_percent, phase3_worse=phase3_percent+phase4_percent+phase5_percent, phase4_worse=phase4_percent+phase5_percent, phase5_worse=phase5_percent
df['phase2_worse'] = df['phase2_percent'] + df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase3_worse'] = df['phase3_percent'] + df['phase4_percent'] + df['phase5_percent']
df['phase4_worse'] = df['phase4_percent'] + df['phase5_percent']
df['phase5_worse'] = df['phase5_percent']
#drop phase2_percent, phase3_percent, phase4_percent, phase5_percent, phase1_percent
df = df.drop(['phase2_percent', 'phase3_percent', 'phase4_percent', 'phase5_percent', 'phase1_percent'], axis=1)
# Splitting the data
#test_df = df.groupby('area_id').tail(1)
#train_df = df.drop(test_df.index)
#test_df = test_df.drop(['area_id','date'], axis=1)
#train_df = train_df.drop(['area_id','date'], axis=1)



In [4]:
#filter df, area_id is in Nigeria_area['area_id']
df = df[df['area_id'].isin(Nigeria_area['area_id'])]


- Baseline vs B+C only — the pivotal comparison. They use the same discretization rule (th=0.2) but different trained models. If sensitivity, AUC, AP, or mean_phase3_pred move on this row pair, the regression itself improved — that's the result you actually want. If only sensitivity moves but AUC/AP/mean_phase3_pred are flat, B+C didn't help
(look elsewhere).
- A only vs Baseline — both use pred_base; only the threshold tuple changes. AUC, AP, mean_phase3_pred, median_phase3_pred are identical by construction; sensitivity and precision shift, prevalences shift. Pure post-processing effect.
- Rank only vs Baseline — diagnostic. Rank rule fixes prediction prevalence at ≈ q3+q4+q5 ≈ 27%. If sensitivity jumps far above the Baseline, the underlying ranking (AUC) is informative even though the threshold rule's calibration was bad. AUC/AP unchanged from Baseline (continuous predictions are identical).
- A+B+C+Rank — combined upper bound. Use as a ceiling reference, not as a recommendation; rank rule is not canonical IPC.
- q3 sweep (Rank only) — traces the sensitivity/precision frontier; pick the q3 whose pred_phase3plus_prevalence is closest to the year's true_phase3plus_prevalence for the most defensible operating point.

In [ ]:
# === NGA ablation helpers — define once, reuse across test years ===
import math
from ipcch.food_crisis_functions import (
    convert_prob_to_phase,
    convert_prob_to_phase_rank,
    compute_extended_metrics,
)

# Knobs that affect TRAINING. If you change these, training reruns below pick them up.
CRISIS_WEIGHT_MULTIPLIER = 4.0   # lever B: sw = 1 + alpha * (y_train > cutoff)
CRISIS_WEIGHT_CUTOFF     = 0.05  # lever B
P3_QUANTILE_ALPHA        = 0.7   # lever C: quantile_alpha for phase-3 regressor

# Knobs that DO NOT affect training (consumed at scoring time).
PHASE3_THRESHOLD = 0.15
TH_PERPHASE      = (0.20, PHASE3_THRESHOLD, 0.20, 0.20)   # lever A
q_rank           = (0.55, 0.20, 0.05, 0.02)               # rank rule (q2,q3,q4,q5)
Q3_SWEEP         = [0.15, 0.20, 0.25, 0.30]

nga_knobs = {
    "CRISIS_WEIGHT_MULTIPLIER": CRISIS_WEIGHT_MULTIPLIER,
    "CRISIS_WEIGHT_CUTOFF":     CRISIS_WEIGHT_CUTOFF,
    "P3_QUANTILE_ALPHA":        P3_QUANTILE_ALPHA,
}

# Six-run grid: (label, predictions_key, mode, knob).
# B+C only vs Baseline isolates training-side improvement from post-processing
# relabeling; AUC/AP and mean/median phase3_pred are continuous-prediction
# diagnostics that discretization rules cannot move. Rank only / *+Rank rows
# are diagnostics: rank rule fixes prediction prevalence by construction and
# is not a canonical IPC conversion.
ABLATION_RUNS = [
    ("Baseline",    "base", "threshold", 0.2),
    ("A only",      "base", "threshold", TH_PERPHASE),
    ("B+C only",    "bc",   "threshold", 0.2),
    ("A+B+C",       "bc",   "threshold", TH_PERPHASE),
    ("Rank only",   "base", "rank",      q_rank),
    ("A+B+C+Rank",  "bc",   "rank",      q_rank),
]

NUM_COLS = ["accuracy","sensitivity","precision","r2","true_phase3plus_prevalence",
            "pred_phase3plus_prevalence","mean_phase3_pred","median_phase3_pred",
            "auc_phase3plus","ap_phase3plus"]


def _train_nga_variant(df, train_date, cutoff_date, params_base_in, params_p3_in,
                       use_bc=False,
                       sw_alpha=CRISIS_WEIGHT_MULTIPLIER,
                       sw_cutoff=CRISIS_WEIGHT_CUTOFF,
                       p3_quantile_alpha=P3_QUANTILE_ALPHA):
    """Train 4 phase regressors. Returns (y_pred_test_long, quantile_objective_used).

    use_bc=False reproduces the original lever-free behavior (phase 2 uses
    params_base, phases 3-5 use params_p3, no sample weights). use_bc=True
    applies lever B (sample weights on all 4 fits) and lever C (quantile loss
    on the phase-3 fit only).
    """
    quantile_objective_used = False
    y_pred_test = pd.DataFrame()
    cur_params = dict(params_base_in)  # phase 2 path
    for i in range(2, 6):
        train_df_loc = df[df["date"] < train_date]
        if cutoff_date is None:
            test_df_loc = df[df["date"] >= train_date]
        else:
            test_df_loc = df[(df["date"] < cutoff_date) & (df["date"] >= train_date)]
        train_df_loc = train_df_loc.drop(["date", "area_id"], axis=1)
        test_df_loc  = test_df_loc.drop(["date", "area_id"], axis=1)
        cols_drop = ["phase{}_worse".format(j) for j in range(2, 6) if j != i]
        train_df_new = train_df_loc.drop(cols_drop, axis=1).dropna(
            subset=["phase{}_worse".format(i)])
        test_df_new  = test_df_loc.drop(cols_drop, axis=1).dropna(
            subset=["phase{}_worse".format(i)])
        test_index = test_df_new.index
        X_train = train_df_new.drop("phase{}_worse".format(i), axis=1)
        y_train = train_df_new["phase{}_worse".format(i)]
        X_test  = test_df_new.drop("phase{}_worse".format(i), axis=1)
        y_test  = test_df_new["phase{}_worse".format(i)]

        # Match original: phase 2 uses base params, phases 3-5 use p3 params.
        if i == 3:
            cur_params = dict(params_p3_in)

        params = dict(cur_params)
        applied_quantile_here = bool(use_bc and i == 3)
        if applied_quantile_here:
            params["objective"]      = "reg:quantileerror"
            params["quantile_alpha"] = p3_quantile_alpha

        sw = (1.0 + sw_alpha * (y_train > sw_cutoff).astype(float)) if use_bc else None

        try:
            model = xgb.XGBRegressor(**params)
            if sw is not None:
                model.fit(X_train, y_train, sample_weight=sw)
            else:
                model.fit(X_train, y_train)
            if applied_quantile_here:
                quantile_objective_used = True
        except Exception as e:
            # Older XGBoost may reject reg:quantileerror; fall back without it.
            print("WARNING: phase {} fit with quantile loss failed ({}: {}); "
                  "falling back to default objective."
                  .format(i, type(e).__name__, e))
            params = dict(cur_params)
            model = xgb.XGBRegressor(**params)
            if sw is not None:
                model.fit(X_train, y_train, sample_weight=sw)
            else:
                model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        row = {"y_pred": y_pred, "y_test": y_test, "phase": [i] * len(y_pred)}
        if i == 5:
            row["test_index"] = test_index
        y_pred_test = pd.concat([y_pred_test, pd.DataFrame(row)], ignore_index=True)
    return y_pred_test, quantile_objective_used


def _score_one(label, src_df, mode, knob, has_bc, qou, test_year):
    if mode == "threshold":
        out = convert_prob_to_phase(src_df.copy(), th=knob)
        th_label, q_label = repr(knob), ""
    else:
        out = convert_prob_to_phase_rank(src_df.copy(), q=knob)
        th_label, q_label = "", repr(knob)
    m = compute_extended_metrics(out)
    return {
        "run": label, "test_year": test_year, **m,
        "th": th_label, "q_rank": q_label,
        "sw_alpha":  nga_knobs["CRISIS_WEIGHT_MULTIPLIER"] if has_bc else "",
        "sw_cutoff": nga_knobs["CRISIS_WEIGHT_CUTOFF"]     if has_bc else "",
        "quantile_alpha": nga_knobs["P3_QUANTILE_ALPHA"]   if has_bc else "",
        "quantile_objective_used": bool(qou) if has_bc else False,
    }


def run_year_ablation(df, test_year, train_date, cutoff_date,
                      params_base, params_p3,
                      results_path="Temp_results.txt"):
    """Train both variants for one test year, score the ablation grid + q3 sweep,
    print the table, and append it to results_path. Returns the results DataFrame."""
    y_pred_test_base, _   = _train_nga_variant(
        df, train_date, cutoff_date, params_base, params_p3, use_bc=False)
    y_pred_test_bc,   qou = _train_nga_variant(
        df, train_date, cutoff_date, params_base, params_p3, use_bc=True)
    print("test_year={}: variants cached. quantile_objective_used={}".format(
        test_year, qou))

    preds = {"base": y_pred_test_base, "bc": y_pred_test_bc}
    rows = [
        _score_one(name, preds[key], mode, knob,
                   has_bc=(key == "bc"),
                   qou=qou if key == "bc" else False,
                   test_year=test_year)
        for name, key, mode, knob in ABLATION_RUNS
    ]
    # q3 sweep — Rank only on pred_base, no retraining.
    rows.extend(
        _score_one("Rank only q3={:.2f}".format(q3),
                   y_pred_test_base, "rank",
                   (q_rank[0], q3, q_rank[2], q_rank[3]),
                   has_bc=False, qou=False, test_year=test_year)
        for q3 in Q3_SWEEP
    )

    results_df = pd.DataFrame(rows)
    for c in NUM_COLS:
        results_df[c] = pd.to_numeric(results_df[c], errors="coerce").round(4)
    print(results_df.to_string(index=False))

    with open(results_path, "a") as f:
        f.write("\n# NGA ablation  test_year={}\n".format(test_year))
        f.write(results_df.to_string(index=False))
        f.write("\n")
    print("\nAppended {} rows to {}".format(len(results_df), results_path))
    return results_df


In [ ]:
# === NGA ablation — test_year=2024 ===
results_2024 = run_year_ablation(
    df, test_year=2024, train_date="2024-01-01", cutoff_date=None,
    params_base=best_params_xgb_regressor,
    params_p3=best_params_xgb_regressor_for_p3,
)


In [ ]:
# === NGA ablation — test_year=2023 ===
results_2023 = run_year_ablation(
    df, test_year=2023, train_date="2023-01-01", cutoff_date="2024-01-01",
    params_base=best_params_xgb_regressor,
    params_p3=best_params_xgb_regressor_for_p3,
)


In [ ]:
# === NGA ablation — test_year=2022 ===
results_2022 = run_year_ablation(
    df, test_year=2022, train_date="2022-01-01", cutoff_date="2023-01-01",
    params_base=best_params_xgb_regressor,
    params_p3=best_params_xgb_regressor_for_p3,
)
